<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 01. Criterios de División: Gini vs Entropía
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/01_Criterios_Division_y_Arboles_Clasificacion_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🍬

En el cuaderno 00 conociste el juego de las 20 preguntas: el árbol le hace preguntas de Sí/No a los datos hasta llegar a una respuesta. Pero, ¿cómo elige el árbol *cuál* pregunta hacer primero, y en qué orden?

Este cuaderno es la versión sencilla del módulo 01 (Criterios de División y Árboles de Clasificación). Si el cuaderno principal habló de "Gini", "Entropía" y "Ganancia de Información" y te sonó a clase de estadística avanzada, aquí lo vemos con bolsas de dulces y ejemplos que puedes correr tú mismo.

Al terminar podrás explicar, con tus propias palabras:
1. Qué significa que un grupo de datos sea "puro" o "impuro".
2. Cómo el índice de Gini mide esa impureza con un número.
3. Que Entropía es otra forma (parecida) de medir lo mismo.
4. Que el árbol prueba muchas preguntas posibles y se queda con la que deja los grupos más puros.

---
## 1. La Bolsa de Dulces: midiendo qué tan "revuelto" está un grupo 🍬

Imagina que tienes una bolsa de dulces y quieres describir qué tan mezclada está:

- **Bolsa A:** 10 chocolates, 0 gomitas. Si metes la mano y sacas un dulce, sabes con certeza que es chocolate. Está **totalmente pura**.
- **Bolsa B:** 5 chocolates, 5 gomitas. Si metes la mano, no tienes ni idea de qué vas a sacar. Está **totalmente revuelta (impura)**.

El **índice de Gini** es solo un número entre 0 y 0.5 (con dos categorías) que resume esta idea:

$$\text{Gini} = 1 - (p_{choco}^2 + p_{gomita}^2)$$

- Bolsa 100% de un solo tipo $\to$ Gini = 0 (pureza total).
- Bolsa mitad y mitad $\to$ Gini = 0.5 (máxima mezcla).

Cuando el árbol de decisión hace una pregunta y divide los datos en dos grupos, calcula el Gini de cada grupo resultante. **Entre más bajo el Gini, mejor fue la pregunta**, porque separó mejor las categorías.

In [ ]:
def gini(chocolates, gomitas):
    total = chocolates + gomitas
    if total == 0:
        return 0.0
    p_choco = chocolates / total
    p_gomita = gomitas / total
    return 1 - (p_choco**2 + p_gomita**2)

# Bolsa A: totalmente pura (solo chocolates)
print('Bolsa A (10 choco, 0 gomitas)  -> Gini =', gini(10, 0))

# Bolsa B: totalmente revuelta (mitad y mitad)
print('Bolsa B (5 choco, 5 gomitas)   -> Gini =', gini(5, 5))

# Bolsa C: un caso intermedio
print('Bolsa C (8 choco, 2 gomitas)   -> Gini =', gini(8, 2))

### 🤔 ¿Qué acaba de pasar?

- La función `gini(...)` es exactamente la fórmula de arriba, aplicada a conteos de dulces en vez de fórmulas abstractas.
- La Bolsa A (pura) dio Gini = 0.0, tal como esperábamos: no hay sorpresa posible.
- La Bolsa B (mitad y mitad) dio Gini = 0.5: la máxima impureza posible con dos categorías.
- La Bolsa C, con 8 de 10 chocolates, quedó en un punto intermedio: bastante pura, pero no perfecta.
- Cuando el árbol decide cuál pregunta hacer, en el fondo está comparando bolsas como estas: prueba muchas preguntas posibles y elige la que deja las bolsas resultantes lo más puras posible (Gini más bajo).

---
## 2. ¿Gini o Entropía? Dos reglas casi gemelas 📏

Existe una segunda forma de medir qué tan revuelta está una bolsa: la **Entropía**, que viene de la teoría de la información (la misma idea que se usa para comprimir archivos). En vez de elevar las proporciones al cuadrado, usa logaritmos:

$$H = -\sum p_k \log_2(p_k)$$

No necesitas memorizar la fórmula. Lo importante es la intuición: Entropía también vale 0 cuando la bolsa es pura, y es máxima cuando está mitad y mitad — igual que Gini, solo que con una "curva" ligeramente distinta.

El algoritmo que arma el árbol (llamado **CART**) hace lo mismo sin importar cuál regla uses: en cada nodo prueba **todas** las variables y **todos** los posibles puntos de corte, calcula la impureza resultante con Gini o Entropía, y se queda con la pregunta que más la reduce. Luego repite el proceso dentro de cada grupo resultante, una y otra vez, hasta que las hojas quedan (casi) puras.

Vamos a comprobar en un ejemplo real que ambas reglas casi siempre coinciden en la práctica.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

vino = load_wine()

arbol_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
arbol_entropia = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)

exactitud_gini = cross_val_score(arbol_gini, vino.data, vino.target, cv=5).mean()
exactitud_entropia = cross_val_score(arbol_entropia, vino.data, vino.target, cv=5).mean()

print(f'Exactitud con Gini:      {exactitud_gini*100:.2f}%')
print(f'Exactitud con Entropía:  {exactitud_entropia*100:.2f}%')

### 🤔 ¿Qué acaba de pasar?

- Usamos `load_wine()`, un dataset de juguete de Scikit-Learn con mediciones químicas de vinos de 3 tipos distintos.
- Entrenamos dos árboles idénticos en todo excepto en la regla de impureza: uno con `criterion='gini'` y otro con `criterion='entropy'`.
- `cross_val_score(..., cv=5)` entrena y evalúa el árbol 5 veces con distintos pedazos de datos, y promedia la exactitud — así el resultado es más confiable que probar una sola vez.
- En la práctica, ambos criterios suelen dar una exactitud casi idéntica. Gini es un poco más rápido de calcular (no usa logaritmos), por eso Scikit-Learn lo usa por defecto.

---
##### 🎯 Reto Práctico para Dummies: ¿Cuál bolsa es más pura?

Tienes dos bolsas de caramelos:
- **Bolsa X:** 12 caramelos de menta, 3 de limón.
- **Bolsa Y:** 9 caramelos de menta, 9 de limón.

¿Cuál bolsa tiene menor Gini (más pureza)? Usa la función `gini(...)` que ya definimos arriba para comprobarlo.

In [ ]:
# =========================================================================
# TU SOLUCIÓN: Reto Dummies 1 - Comparando la pureza de dos bolsas
# =========================================================================

# gini_x = gini(12, 3)
# gini_y = gini(9, 9)
# print(...)


<details>
<summary><b>💡 Haz clic aquí para ver la solución explicada...</b></summary>

```python
gini_x = gini(12, 3)
gini_y = gini(9, 9)

print(f'Bolsa X (12 menta, 3 limón) -> Gini = {gini_x:.3f}')
print(f'Bolsa Y (9 menta, 9 limón)  -> Gini = {gini_y:.3f}')
print('👉 La Bolsa X es más pura porque tiene un Gini más bajo (está menos mezclada).')
```
</details>

---
## 3. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Impureza | Qué tan mezclada está una bolsa (o un nodo) en cuanto a categorías. |
| Gini | Número entre 0 y 0.5 que mide impureza: 0 = pura, 0.5 = totalmente mezclada. |
| Entropía | Otra forma de medir impureza, basada en logaritmos; casi siempre da resultados parecidos a Gini. |
| CART | El algoritmo que prueba todas las preguntas posibles en cada nodo y elige la que más reduce la impureza. |
| Ganancia de Información | Cuánto "bajó" la impureza gracias a una pregunta (entre más alta, mejor la pregunta). |

➡️ **Siguiente paso:** en el cuaderno [02 - Árboles de Regresión y Poda (Para Dummies)](02_Arboles_Regresion_y_Poda_Cost_Complexity_Dummies.ipynb) veremos qué pasa cuando en lugar de categorías (Sí/No) queremos predecir un número, y cómo evitar que el árbol se aprenda los datos de memoria.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
